# 01 · Ingest and clean

Four raw CSVs in, seven parquet frames out. The only notebook that touches
the zip; everything downstream reads `clean/`.

**Run this once.** Re-run it only when `src/etl.py` changes.

In [1]:
%load_ext autoreload
%autoreload 2

import pathlib
import sys

_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "etl.py").exists())
sys.path.insert(0, str(_root / "src"))

from nbinit import *

## 1. Find the files

Filenames drift between the brief, the induction slides and the shared drive
, one of them spells it `attirition_log.csv`. `etl.discover` matches on a
regex per table rather than an exact name, and skips the `__MACOSX` junk that
rides along in zips made on a Mac.

In [2]:
paths = etl.discover(DATA_ZIP)

for key, path in paths.items():
    print(f"  {key:<12} {path.relative_to(ROOT)}")

  employees    extracted\employees.csv
  attrition    extracted\attrition_log.csv
  engagement   extracted\engagement.csv
  performance  extracted\performance.csv


## 2. Load raw

Column names are lowercased and snake_cased on the way in. Nothing else is
changed yet: the point of this step is to see the data as delivered.

In [3]:
raw = etl.load_raw(DATA_ZIP)

assert len(raw) == 4, f"expected 4 tables, found {sorted(raw)}"

  loaded employees    employees.csv                 13,403 rows x 24 cols
  loaded attrition    attrition_log.csv              1,400 rows x 10 cols


  loaded engagement   engagement.csv                55,971 rows x 12 cols
  loaded performance  performance.csv               34,979 rows x  7 cols


### What each table actually contains

In [4]:
for name, frame in raw.items():
    print(f"\n{name.upper()}   {frame.shape[0]:,} rows x {frame.shape[1]} cols")
    print(f"\n")
    print("  " + ", ".join(frame.columns))


EMPLOYEES   13,403 rows x 24 cols


  employee_id, name, hire_date, exit_date, status, department, role_family, role_level, job_title, salary, compa_ratio, gender, age_band, cultural_background, contract_type, hipo_flag, promotion_eligible, manager_id, hire_source, legacy_entity_code, data_source_system, days_to_fill, tenure_months, acting_appointment

ATTRITION   1,400 rows x 10 cols


  employee_id, exit_date, exit_type, stated_exit_reason, notice_period_served, regrettable_flag, performance_band_at_exit, salary_at_exit, manager_id_at_exit, pathway

ENGAGEMENT   55,971 rows x 12 cols


  employee_id, wave_number, survey_date, response_flag, manager_effectiveness, psychological_safety, recognition, career_development, senior_leadership_trust, purpose_meaning, wellbeing, confidence_in_role_future

PERFORMANCE   34,979 rows x 7 cols


  employee_id, review_date, performance_rating, review_cycle, promotion_recommendation, goal_achievement_score, reviewer_id


In [5]:
raw["employees"].head(3)

,employee_id,name,hire_date,exit_date,status,department,role_family,role_level,job_title,salary,compa_ratio,gender,age_band,cultural_background,contract_type,hipo_flag,promotion_eligible,manager_id,hire_source,legacy_entity_code,data_source_system,days_to_fill,tenure_months,acting_appointment
0,E00001,Madison Lang,1988-01-11,NaN,active,Wealth Management,Operations-Processing,2,"Manager, Wealth Management","115,200.00",0.79,Female,60+,Anglo-Australian,Full-time,False,False,E04330,referral,NovaCorp-Origin,WorkdayHR,22.00,462,False
1,E00003,Amy Garza,1988-01-13,NaN,active,Wealth Management,Corporate-Support,3,"Senior Manager, Wealth Management","170,200.00",0.85,Female,60+,South Asian,Full-time,False,False,E13109,direct,NovaCorp-Origin,WorkdayHR,62.00,462,False
2,E00004,Meghan Robertson,1988-01-14,NaN,active,Insurance,Management,2,"Manager, Insurance","125,500.00",0.87,Female,60+,East Asian,Full-time,False,False,E14294,referral,NovaCorp-Origin,WorkdayHR,86.00,462,False


In [6]:
raw["attrition"].head(3)

,employee_id,exit_date,exit_type,stated_exit_reason,notice_period_served,regrettable_flag,performance_band_at_exit,salary_at_exit,manager_id_at_exit,pathway
0,E00005,2024-12-24,voluntary,Relocation,True,False,Meets Expectations,"132,400.00",E12337,push
1,E00006,2025-04-17,voluntary,Work-life balance,True,False,Below Expectations,"120,600.00",E00218,push
2,E00008,2024-02-19,voluntary,Career advancement,True,False,Meets Expectations,"117,800.00",E00249,push


In [7]:
raw["engagement"].head(3)

,employee_id,wave_number,survey_date,response_flag,manager_effectiveness,psychological_safety,recognition,career_development,senior_leadership_trust,purpose_meaning,wellbeing,confidence_in_role_future
0,E00001,1,2024-03-08,True,5.00,4.39,4.55,4.25,3.72,3.47,4.83,4.57
1,E00001,2,2024-06-27,True,4.69,4.26,4.10,3.52,3.57,4.00,4.79,4.71
2,E00001,3,2024-10-25,True,4.65,4.67,4.87,3.72,3.51,3.80,4.52,4.34


In [8]:
raw["performance"].head(3)

,employee_id,review_date,performance_rating,review_cycle,promotion_recommendation,goal_achievement_score,reviewer_id
0,E00001,2024-06-01,Meets Expectations,2024-H1,False,46.10,E04330
1,E00001,2024-11-21,Meets Expectations,2024-H2,False,65.40,E04330
2,E00001,2025-05-28,Meets Expectations,2025-H1,False,76.40,E04330


## 3. Coerce types

CSV gives everything as `object`. `etl.clean_all` applies a per-table spec:
dates ISO first then day-first (the exports mix both), booleans from the
several spellings of "true", numerics with currency symbols stripped.

Categoricals are trimmed but **deliberately not case-folded**: `"Agency"`
next to `"agency"` is a data quality finding, and folding it silently would
hide the legacy-system merge that caused it.

In [9]:
clean = etl.clean_all(raw)

before_after = pd.DataFrame({
    "raw": raw["employees"].dtypes.astype(str),
    "clean": clean["employees"].dtypes.astype(str),
})
before_after[before_after["raw"] != before_after["clean"]]

,raw,clean
employee_id,str,string
hire_date,str,datetime64[us]
exit_date,str,datetime64[us]
status,str,string
department,str,string
role_family,str,string
job_title,str,string
gender,str,string
age_band,str,string
cultural_background,str,string


### Did any date fail to parse?

`to_date` coerces failures to `NaT`, so a format problem shows up as a jump
in nulls rather than an exception.

In [10]:
for table, cols in [("employees", ["hire_date", "exit_date"]),
                    ("attrition", ["exit_date"]),
                    ("engagement", ["survey_date"]),
                    ("performance", ["review_date"])]:
    for col in cols:
        if col in clean[table].columns:
            raw_present = raw[table][col].notna().sum()
            parsed = clean[table][col].notna().sum()
            flag = "" if raw_present == parsed else "   <-- LOST ROWS"
            print(f"  {table:<12} {col:<14} present {raw_present:>7,}  parsed {parsed:>7,}{flag}")

  employees    hire_date      present  13,403  parsed  13,403
  employees    exit_date      present   1,400  parsed   1,400
  attrition    exit_date      present   1,400  parsed   1,400
  engagement   survey_date    present  55,971  parsed  55,971
  performance  review_date    present  34,979  parsed  34,979


## 4. Build the analysis frame

One row per employee, with attrition detail, engagement behaviour,
performance history and manager context joined on. The merge is validated
`one_to_one`: if the attrition log ever gains a duplicate `employee_id`,
this raises rather than quietly fanning the roster out.

In [11]:
df = etl.build_analysis_df(clean)

print(f"  analysis_df  {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"  one row per employee: {df['employee_id'].is_unique}")

  analysis_df  13,403 rows x 87 cols
  one row per employee: True


### Two supporting shapes

`engagement_long` keeps one row per employee-per-wave, with employee
attributes attached: the shape time-series plots and facet grids need.
`monthly_exits` pre-aggregates exits by month, department and FAR wave for
the event study in `04`.

In [12]:
eng_long = etl.engagement_long(clean, df)
monthly = etl.monthly_exits(df)

print(f"  engagement_long  {eng_long.shape[0]:,} rows x {eng_long.shape[1]} cols")
print(f"  monthly_exits    {monthly.shape[0]:,} rows x {monthly.shape[1]} cols")
print(f"  months covered   {monthly['month'].min():%b %Y} to {monthly['month'].max():%b %Y}")

  engagement_long  55,971 rows x 30 cols
  monthly_exits    153 rows x 10 cols
  months covered   Jan 2024 to Dec 2025


### What got derived

24 raw employee columns became 87. Grouped by what they are for:

In [13]:
derived = [c for c in df.columns if c not in clean["employees"].columns]

rules = [
    ("attrition detail (joined)", lambda c: c in clean["attrition"].columns),
    ("engagement behaviour", lambda c: c.startswith(("composite", "waves", "response", "never", "went", "trailing", "persistently", "eng_"))),
    ("performance history", lambda c: c.startswith(("rating", "goal", "n_reviews", "n_promo", "ever_promo", "top_talent", "last_review"))),
    ("manager context", lambda c: c.startswith(("span_", "team_"))),
    ("FAR (angle 2)", lambda c: c.startswith(("far_", "months_to_far", "exited_post"))),
    ("cost (angle 1)", lambda c: "_cost" in c or c.endswith("_premium") or c.startswith("regrettable_")),
]

# First matching rule wins, so nothing is listed twice.
remaining, groups = list(derived), {}
for label, matches in rules:
    groups[label] = [c for c in remaining if matches(c)]
    remaining = [c for c in remaining if c not in groups[label]]
groups["shared cuts"] = remaining

for label, cols in groups.items():
    print(f"\n  {label}  ({len(cols)})")
    print("    " + ", ".join(cols))


  attrition detail (joined)  (8)
    exit_type, stated_exit_reason, notice_period_served, regrettable_flag, performance_band_at_exit, salary_at_exit, manager_id_at_exit, pathway

  engagement behaviour  (22)
    waves_issued, waves_responded, composite_mean, composite_first, composite_last, composite_min, composite_std, response_rate, never_responded, composite_trend, trailing_silence, went_silent, waves_low, persistently_disengaged, eng_manager_effectiveness, eng_psychological_safety, eng_recognition, eng_career_development, eng_senior_leadership_trust, eng_purpose_meaning, eng_wellbeing, eng_confidence_in_role_future

  performance history  (11)
    n_reviews, rating_mean, rating_last, rating_first, goal_mean, goal_last, n_promo_recommended, last_review_date, rating_trend, ever_promo_recommended, top_talent

  manager context  (3)
    span_of_control, team_departures, team_attrition_rate

  FAR (angle 2)  (6)
    far_date, far_wave, far_exposed, far_exposed_senior, months_to_far_at_

## 5. Cache to parquet

Parquet keeps the dtypes CSV would flatten to strings, and reloads in under
a second.

In [14]:
for frame, name in [(clean["employees"], "employees_clean"),
                    (clean["attrition"], "attrition_clean"),
                    (clean["engagement"], "engagement_clean"),
                    (clean["performance"], "performance_clean"),
                    (df, "analysis_df"),
                    (eng_long, "engagement_long"),
                    (monthly, "monthly_exits")]:
    cache(frame, name)

  cached employees_clean       13,403 rows x  24 cols  (0.5 MB)
  cached attrition_clean        1,400 rows x  10 cols  (0.0 MB)


  cached engagement_clean      55,971 rows x  12 cols  (0.7 MB)
  cached performance_clean     34,979 rows x   7 cols  (0.3 MB)
  cached analysis_df           13,403 rows x  87 cols  (1.4 MB)


  cached engagement_long       55,971 rows x  30 cols  (1.0 MB)
  cached monthly_exits            153 rows x  10 cols  (0.0 MB)


### Round-trip check

Read back what was just written, so an unrepresentable dtype fails here
rather than three notebooks downstream.

In [15]:
for name in CACHED_FRAMES:
    frame = load(name)
    print(f"  {name:<20} {frame.shape[0]:>7,} rows x {frame.shape[1]:>3} cols   ok")

  employees_clean       13,403 rows x  24 cols   ok
  attrition_clean        1,400 rows x  10 cols   ok
  engagement_clean      55,971 rows x  12 cols   ok
  performance_clean     34,979 rows x   7 cols   ok
  analysis_df           13,403 rows x  87 cols   ok
  engagement_long       55,971 rows x  30 cols   ok
  monthly_exits            153 rows x  10 cols   ok


## Next

**`02_data_quality.ipynb`**: profile the tables and reconcile them against
the annual report. Do not build anything on top of this frame before reading
that one: the headline numbers do not agree, and which one is wrong changes
the answer to every angle.